In [ ]:
%matplotlib qt

import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
import numpy as np
import pandas as pd
import os
import spectral.io.envi as envi
from tkinter import Tk, filedialog

class DataCubeBrowser:
    def __init__(self, filepath=None, rgb_bands=(30, 20, 10), max_display_size=1000):
        self.filepath = filepath
        self.rgb_bands = rgb_bands
        self.max_display_size = max_display_size
        
        self.src = None
        self.wavelengths = None
        self.img_rgb = None

        if filepath:
            self.load_file(filepath)

    def load_file(self, filepath):
        self.filepath = filepath

        self.wavelengths = None
        hdr_file = filepath.replace('.bsq', '.hdr')
        if not os.path.exists(hdr_file):
            hdr_file = filepath + '.hdr'
            
        if os.path.exists(hdr_file):
            try:
                hdr = envi.read_envi_header(hdr_file)
                if 'wavelength' in hdr:
                    self.wavelengths = np.array([float(w) for w in hdr['wavelength']])
            except Exception as e:
                print('Could not read ENVI header:', e)
                
        self.src = rasterio.open(filepath)
        
        if self.wavelengths is None:
            self.wavelengths = np.arange(1, self.src.count + 1)
            
        print(f'Opened {os.path.basename(filepath)}')
        print(f'Dimensions: {self.src.width}x{self.src.height}, Bands: {self.src.count}')
        
        self._load_rgb()
        
        if hasattr(self, 'fig'):
            self.refresh_display()

    def _load_rgb(self):
        factor = max(1, max(self.src.width, self.src.height) // self.max_display_size)
        self.preview_width = self.src.width // factor
        self.preview_height = self.src.height // factor
        self.scale_factor = factor
        
        r = self.src.read(self.rgb_bands[0], out_shape=(self.preview_height, self.preview_width))
        g = self.src.read(self.rgb_bands[1], out_shape=(self.preview_height, self.preview_width))
        b = self.src.read(self.rgb_bands[2], out_shape=(self.preview_height, self.preview_width))
        
        def stretch(band):
            p2, p98 = np.nanpercentile(band, (2, 98))
            if (p98 - p2) == 0:
                return np.zeros_like(band)
            return np.clip((band - p2) / (p98 - p2), 0, 1)
            
        self.img_rgb = np.dstack([stretch(r), stretch(g), stretch(b)])

    def show(self):
        self.fig = plt.figure(figsize=(14, 6))

        self.ax_img = self.fig.add_axes([0.05, 0.25, 0.4, 0.7])
        self.ax_img.axis('off')

        self.ax_plot = self.fig.add_axes([0.55, 0.25, 0.4, 0.7])
        self.ax_plot.set_title('Spectral Signature')
        self.ax_plot.set_xlabel('Wavelength / Band')
        self.ax_plot.set_ylabel('Reflectance')
        self.ax_plot.grid(True)

        self.line, = self.ax_plot.plot([], [])

        self.current_x = None
        self.current_y = None
        self.current_signature = None

        self.fig.canvas.mpl_connect('button_press_event', self.onclick)

        ax_btn_load = self.fig.add_axes([0.1, 0.05, 0.15, 0.1])
        ax_btn_save = self.fig.add_axes([0.3, 0.05, 0.15, 0.1])

        self.btn_load = Button(ax_btn_load, 'Wczytaj plik')
        self.btn_save = Button(ax_btn_save, 'Zapisz CSV')

        for btn in [self.btn_load, self.btn_save]:
            btn.color = '#2c3e50'
            btn.hovercolor = '#34495e'

        self.btn_load.on_clicked(self.open_file_dialog)
        self.btn_save.on_clicked(self.export_csv)

        plt.show()

        if self.filepath:
            self.refresh_display()

    def refresh_display(self):
        if self.img_rgb is None:
            return

        self.ax_img.clear()
        self.ax_img.imshow(self.img_rgb)
        self.ax_img.set_title(os.path.basename(self.filepath))
        self.ax_img.axis('off')

        self.ax_plot.clear()
        self.ax_plot.set_title('Spectral Signature')
        self.ax_plot.set_xlabel('Wavelength / Band')
        self.ax_plot.set_ylabel('Reflectance')
        self.ax_plot.grid(True)

        self.line, = self.ax_plot.plot([], [])

        self.current_signature = None
        self.current_x = None
        self.current_y = None

        self.fig.canvas.draw_idle()

    def open_file_dialog(self, event):
        root = Tk()
        root.withdraw()
        filepath = filedialog.askopenfilename(
            title="Wybierz plik BSQ",
            filetypes=[("BSQ files", "*.bsq"), ("All files", "*.*")]
        )
        root.destroy()

        if filepath:
            self.load_file(filepath)

    def onclick(self, event):
        if event.inaxes != self.ax_img or self.src is None:
            return
            
        px = int(event.xdata)
        py = int(event.ydata)
        
        x_orig = int(px * self.scale_factor)
        y_orig = int(py * self.scale_factor)
        
        x_orig = np.clip(x_orig, 0, self.src.width - 1)
        y_orig = np.clip(y_orig, 0, self.src.height - 1)
        
        print(f'Pixel: X={x_orig}, Y={y_orig}')
        
        w = Window(col_off=x_orig, row_off=y_orig, width=1, height=1)
        signature = self.src.read(window=w).flatten()
        
        self.line.set_data(self.wavelengths, signature)
        self.ax_plot.relim()
        self.ax_plot.autoscale_view()
        self.ax_plot.set_title(f'Signature (X:{x_orig}, Y:{y_orig})')
        self.fig.canvas.draw_idle()
        
        self.current_x = x_orig
        self.current_y = y_orig
        self.current_signature = signature

    def export_csv(self, event):
        if self.current_signature is None:
            print("Najpierw kliknij piksel!")
            return
            
        filename = f'signature_X{self.current_x}_Y{self.current_y}.csv'
        
        df = pd.DataFrame({
            'Wavelength': self.wavelengths,
            'Reflectance': self.current_signature
        })
        
        df.to_csv(filename, index=False)


In [41]:
browser = DataCubeBrowser(bsq_file, rgb_bands=(60, 40, 20))
browser.show()

Opened 221000_Odra_HS_Blok_A_015_VS_join_atm.bsq
Dimensions: 3862x2838, Bands: 456
